# Word Embeddings

Dans cette démonstration, nous allons voir comment utiliser des word embeddings pré-appris avec [`spacy`](https://spacy.io/) et [`gensim`](https://radimrehurek.com/gensim/).

Nous verrons ensuite comment apprendre nos propres embeddings.

In [1]:
!pip install gensim

  Using cached gensim-4.4.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (8.4 kB)
Using cached gensim-4.4.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (27.8 MB)


In [2]:
import itertools
import os
import pathlib
import sys
import re
import typing

import gensim.models
import gensim.models.phrases
import spacy
import warnings
warnings.filterwarnings("ignore")

## Modèle pré-entraîné de Spacy

Nous allons travailler avec un petit modèle pour l'anglais, qui contient des embeddings pour 20 000 mots.

In [3]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Utilisation

In [4]:
nlp = spacy.load('en_core_web_sm')

doc = nlp("This is some text that I am processing with Spacy")

print(doc[3].vector)

print("Vecteur moyen")
print(doc.vector)

[-0.04028979 -0.42573857  0.22283356 -0.19462731 -0.18633991  0.03154597
  1.25567     0.50992644 -0.31769255 -0.3719846  -0.4720959  -1.3329258
 -0.6757268   1.4606285   0.2425617   0.70898944 -0.9689184  -0.88235253
 -0.21309108  1.8374306   1.0599436   0.78597385 -0.08130729 -0.21318793
  0.4048365  -0.41736788  0.372233   -0.664698    0.32333145 -0.0164865
 -0.44257432  0.14839518 -0.19505021  0.1781289  -0.2902862   1.1999402
 -0.09980147 -0.01726501  0.30666587 -0.19327636 -0.6077464  -0.5394199
 -0.17761993 -0.35744295 -0.5014699  -0.69022095 -0.06758932  0.25100756
 -0.05754712  0.52696526 -0.2957775  -0.02683009  0.9336513  -0.6980016
  1.1310893  -0.547488    0.74785584  0.61228734  0.03016563 -0.6558356
 -0.9416984   0.4078372  -0.2946821   0.70640457  0.80438346 -0.16768646
 -0.5760308  -0.15040277  0.21358788  0.13232726 -0.3572658  -0.0351385
  0.8047565  -1.3231521  -0.08712146  0.08009386  0.03687882 -0.7006719
  0.41156214 -0.23079246  0.18822306 -0.24170646 -0.6328960

## Modèle pré-entraîné de Gensim

Gensim ne fournit pas d'embeddings pré-entraînés, mais un mécanisme pour en charger.

Quelques sources d'embeddings pré-entraînés :

- [Documentation word2vec de Google News](https://code.google.com/archive/p/word2vec/) ; [GoogleNews-vectors-negative300.bin.gz](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)

- [Dépôt de word embeddings du NLPL](http://vectors.nlpl.eu/repository/)

- [fasttext](https://fasttext.cc/docs/en/crawl-vectors.html)

- [Embeddings français de Jean-Philippe Fauconnier](https://fauconnier.github.io/)

In [5]:
!wget https://embeddings.net/embeddings/frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin

--2026-10-08 20:22:19--  https://embeddings.net/embeddings/frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin
Resolving embeddings.net (embeddings.net)... 93.127.179.204, 77.37.53.122, 2a02:4780:24:148f:6b35:2733:abaa:7042, ...
Connecting to embeddings.net (embeddings.net)|93.127.179.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126052447 (120M) [application/octet-stream]
Saving to: ‘frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin’

frWac_non_lem_no_po 100%[===================>] 120.21M  53.1MB/s    in 2.3s    

2026-10-08 20:22:21 (53.1 MB/s) - ‘frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin’ saved [126052447/126052447]



In [6]:
!ls -l 
model_file = "frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin"

total 123112
-rw-r--r-- 1 root root 126052447 Mar 19  2022 frWac_non_lem_no_postag_no_phrase_200_cbow_cut100.bin
-rw-r--r-- 1 root root     10302 Oct  8 20:21 nlp__word-embeddings__demo-fr.full.ipynb


## Utilisation de [`gensim.models.KeyedVectors`](https://radimrehurek.com/gensim/models/keyedvectors.html)

In [7]:
model = gensim.models.KeyedVectors.load_word2vec_format(model_file, binary=True)
print("Taille du vocabulaire : ",len(model.key_to_index))

Taille du vocabulaire :  155562


In [8]:
for i, word in enumerate(model.index_to_key):
  print(word)
  if i > 50:
    break

</s>
de
la
et
le
l'
les
à
des
d'
en
du
un
est
une
pour
dans
par
sur
qui
que
il
au
a
pas
ce
ou
avec
plus
je
vous
ne
se
sont
s'
qu'
n'
aux
on
mais
nous
c'
son
cette
comme
être
elle
tout
j'
ont
si
ces


In [9]:
# Récupère le vecteur d'un mot
vector = model['facile']

# Le prétraitement doit être le même que pendant l'entraînement
words = "Ceci est une phrase transformée à l'aide de Gensim".lower().split(' ')
vectors = [(word, model[word]) for word in words if word in model.key_to_index]
print("Mots transformés :")
for word, vector in vectors:
  print(word)

Mots transformés :
ceci
est
une
phrase
transformée
à
l'aide
de


In [10]:
model.most_similar('berlin')

[('dresde', 0.7241218090057373),
 ('munich', 0.681473433971405),
 ('leipzig', 0.6640617251396179),
 ('hambourg', 0.6593071818351746),
 ('bonn', 0.6533075571060181),
 ('allemagne', 0.6425371766090393),
 ('düsseldorf', 0.6412529945373535),
 ('allemande', 0.6355056166648865),
 ('potsdam', 0.6336090564727783),
 ('strasse', 0.6298079490661621)]

In [11]:
print(model.similarity('berlin', 'munich'))
print(model.similarity('berlin', 'paris'))

0.68147343
0.18069758


## Opérations dans l'espace sémantique

À l'aide de [`gensim.models.keyedvectors.KeyedVectors.similar_by_vector`](https://radimrehurek.com/gensim/models/keyedvectors.html#gensim.models.keyedvectors.KeyedVectors.similar_by_vector), nous allons définir une fonction d'analogie :

`x` est à `y` ce que `z` est à ?

*Ex.* : *Roi* est à *Reine* ce que *Oncle* est à ?

Réponse : *Tante*

In [12]:
def semantic_analogy(x, y, z):
  return model.similar_by_vector(model[y] - model[x] + model[z])

In [13]:
semantic_analogy("homme", "femme", "chien")

[('chien', 0.8191524744033813),
 ('chienne', 0.7883859872817993),
 ('chatte', 0.6961268186569214),
 ('toutou', 0.6323156952857971),
 ('chiot', 0.6281698346138),
 ('adorable', 0.6145821809768677),
 ('sociable', 0.599670946598053),
 ('chiens', 0.5982583165168762),
 ('affectueux', 0.5909204483032227),
 ('chats', 0.5903921127319336)]

## Apprentissage de word embeddings

### Le corpus 20 Newsgroups

20 000 « posts » répartis en 20 thématiques.

In [14]:
!wget https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
!tar xzf 20news-bydate.tar.gz
!ls 20news-bydate-train/

--2026-10-08 20:22:23--  https://github.com/shuuchuu/datasets/raw/refs/heads/main/20newsgroups/20news-bydate.tar.gz
Resolving github.com (github.com)... 140.82.121.4
Connecting to github.com (github.com)|140.82.121.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/shuuchuu/datasets/refs/heads/main/20newsgroups/20news-bydate.tar.gz [following]
--2026-10-08 20:22:24--  https://raw.githubusercontent.com/shuuchuu/datasets/refs/heads/main/20newsgroups/20news-bydate.tar.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14464277 (14M) [application/octet-stream]
Saving to: ‘20news-bydate.tar.gz’

20news-bydate.tar.g 100%[===================>]  13.79M  55.7MB/s    in 0.2s    

2026-10-08 20:22:24 (55.7

In [15]:
!ls 20news-bydate-test/

alt.atheism		  rec.autos	      sci.space
comp.graphics		  rec.motorcycles     soc.religion.christian
comp.os.ms-windows.misc   rec.sport.baseball  talk.politics.guns
comp.sys.ibm.pc.hardware  rec.sport.hockey    talk.politics.mideast
comp.sys.mac.hardware	  sci.crypt	      talk.politics.misc
comp.windows.x		  sci.electronics     talk.religion.misc
misc.forsale		  sci.med


In [16]:
contents = [p.read_text(encoding="latin-1")
            for p in pathlib.Path(".").glob("20news-bydate-*/*/*")]


print(f"{len(contents)} textes récupérés")

18846 textes récupérés


In [17]:
print(contents[1])

From: acooper@mac.cc.macalstr.edu (Turin Turambar, ME Department of Utter Misery)
Subject: STRONG & weak Atheism
Organization: Macalester College
Lines: 14

Did that FAQ ever got modified to re-define strong atheists as not those who
assert the nonexistence of God, but as those who assert that they BELIEVE in 
the nonexistence of God?  There was a thread on this earlier, but I didn't get
the outcome...

-- Adam "No Nickname" Cooper



********************************************************************************
* Adam John Cooper		"Verily, often have I laughed at the weaklings *
* (612) 696-7521		   who thought themselves good simply because  *
* acooper@macalstr.edu				they had no claws."	       *
********************************************************************************



In [18]:
# Nettoyage
def preprocess(content: str) -> typing.List[str]:
  offset = content.find('\n\n')
  if offset > 0:
    content = content[offset + 2:]
  sentences = (re.sub(r'[\!"#$%&\*+,-./:;<=>?@^_`()|~=]', " ", line).split()
               for line in content.splitlines()
               if not line.endswith("writes:"))
  return list(itertools.chain.from_iterable(sentences))


texts = [preprocess(content) for content in contents]

In [19]:
print(texts[0])

['There', 'lies', 'the', 'hypocrisy', 'dude', 'Atheism', 'takes', 'as', 'much', 'faith', 'as', 'theism', 'Admit', 'it', 'Some', 'people', 'might', 'think', 'it', 'takes', 'faith', 'to', 'be', 'an', 'atheist', 'but', 'faith', 'in', 'what', 'Does', 'it', 'take', 'some', 'kind', 'of', 'faith', 'to', 'say', 'that', 'the', 'Great', 'Invisible', 'Pink', 'Unicorn', 'does', 'not', 'exist', 'Does', 'it', 'take', 'some', 'kind', 'of', 'faith', 'to', 'say', 'that', 'Santa', 'Claus', 'does', 'not', 'exist', 'If', 'it', 'does', 'and', 'it', 'may', 'for', 'some', 'people', 'I', 'suppose', 'it', 'certainly', "isn't", 'as', 'big', 'a', 'leap', 'of', 'faith', 'to', 'say', 'that', 'these', 'things', 'and', 'god', 'DO', 'exist', 'I', 'suppose', 'it', 'depends', 'on', 'your', 'notion', 'and', 'definition', 'of', 'faith', 'Besides', 'not', 'believing', 'in', 'a', 'god', 'means', 'one', "doesn't", 'have', 'to', 'deal', 'with', 'all', 'of', 'the', 'extra', 'baggage', 'that', 'comes', 'with', 'it', 'This', 'l

## Détection de groupes de mots avec le Phraser de Gensim

In [20]:
common_terms = ["of", "with", "without", "and", "or", "the", "a"]

phrases = gensim.models.phrases.Phrases(texts,
                                        connector_words=frozenset(common_terms))

phraser = gensim.models.phrases.Phraser(phrases)
phrased_texts = list(phraser[texts])
len(phrased_texts)

18846

In [21]:
model = gensim.models.Word2Vec(
    phrased_texts,
    min_count=3,
    vector_size=200,
    workers=2,
    window=5,
    epochs=30,
).wv

In [22]:
model.most_similar('New_York')

[('City', 0.5330778360366821),
 ('California', 0.48039117455482483),
 ('Houston', 0.46055614948272705),
 ('Times', 0.440825879573822),
 ('Arizona', 0.4382813274860382),
 ('Boston', 0.4333862066268921),
 ('London', 0.43207746744155884),
 ('County', 0.43180525302886963),
 ('Idaho', 0.4312450885772705),
 ('Atlanta', 0.43010401725769043)]

In [23]:
semantic_analogy("wind", "fly", "water")

[('water', 0.7118360996246338),
 ('fly', 0.6340951919555664),
 ('air', 0.44582024216651917),
 ('grain', 0.43104279041290283),
 ('drop', 0.419924795627594),
 ('cut', 0.4114488661289215),
 ('launch', 0.40077775716781616),
 ('generate', 0.4007507860660553),
 ('lead', 0.3988955020904541),
 ('carry', 0.3928753137588501)]